In [1]:
import json, h5py, numpy as np

DATA = "data"

with open(f"{DATA}/vg150/VG-SGG-dicts.json") as f:
    d = json.load(f)
idx_label = {int(k): v for k, v in d['idx_to_label'].items()}
idx_pred = {int(k): v for k, v in d['idx_to_predicate'].items()}

with h5py.File(f"{DATA}/vg150/VG-SGG.h5", "r") as f:
    splits = f['split'][:]
    first_b = f['img_to_first_box'][:]
    last_b = f['img_to_last_box'][:]
    first_r = f['img_to_first_rel'][:]
    last_r = f['img_to_last_rel'][:]
    boxes_all = f['boxes_1024'][:]
    labels_all = f['labels'][:].flatten()
    rels_all = f['relationships'][:]
    preds_all = f['predicates'][:].flatten()

xc, yc, w, h = boxes_all[:,0], boxes_all[:,1], boxes_all[:,2], boxes_all[:,3]
corners = np.column_stack([(xc-w/2).astype(int),(yc-h/2).astype(int),(xc+w/2).astype(int),(yc+h/2).astype(int)])

test_idx = np.where(splits == 2)[0]
obj_counts = last_b - first_b + 1
busy_idx = test_idx[np.argsort(obj_counts[test_idx])[::-1]]

print("busiest test images:")
for i in busy_idx[:10]:
    rs, re = first_r[i], last_r[i] + 1
    n_rels = max(0, re - rs)
    print(f"  img {i}: {obj_counts[i]} objs, {n_rels} rels")


busiest test images:
  img 81665: 60 objs, 1 rels
  img 77245: 58 objs, 20 rels
  img 102219: 56 objs, 43 rels
  img 82092: 55 objs, 30 rels
  img 94066: 51 objs, 16 rels
  img 98034: 49 objs, 32 rels
  img 78509: 48 objs, 19 rels
  img 102540: 47 objs, 18 rels
  img 98831: 47 objs, 20 rels
  img 86695: 47 objs, 21 rels


In [2]:
# objects in busiest image
b = busy_idx[0]
s, e = first_b[b], last_b[b] + 1
boxes = corners[s:e]
lbls = labels_all[s:e]
rs, re = first_r[b], last_r[b] + 1
rels = rels_all[rs:re] if re > rs else []
preds = preds_all[rs:re] if re > rs else []

print(f"img {b}: {len(boxes)} objs, {len(rels)} rels")
for i in range(min(len(boxes), 30)):
    x1, y1, x2, y2 = boxes[i]
    name = idx_label.get(lbls[i], '?')
    print(f"  [{i:2d}] {name:12s} ({x1:4d},{y1:4d})-({x2:4d},{y2:4d}) {x2-x1:3d}x{y2-y1:3d}")

if len(rels) > 0:
    print(f"\nrelationships ({len(rels)}):")
    seen = set()
    for (sid, oid), pid in zip(rels, preds):
        si, oi = sid - s, oid - s
        if 0 <= si < len(boxes) and 0 <= oi < len(boxes):
            key = (si, oi, pid)
            if key not in seen:
                seen.add(key)
                sn = idx_label.get(lbls[si], '?')
                on = idx_label.get(lbls[oi], '?')
                pn = idx_pred.get(pid, '?')
                print(f"  {sn} --{pn}--> {on}")


img 81665: 60 objs, 0 rels
  [ 0] flower       ( 147, 162)-( 191, 201)  44x 39
  [ 1] flower       ( 231, 287)-( 289, 328)  58x 41
  [ 2] flower       ( 242, 346)-( 283, 398)  41x 52
  [ 3] flower       ( 477, 364)-( 531, 412)  54x 48
  [ 4] flower       ( 402, 372)-( 443, 420)  41x 48
  [ 5] flower       ( 338, 401)-( 375, 445)  37x 44
  [ 6] flower       ( 420, 172)-( 459, 220)  39x 48
  [ 7] flower       ( 421, 214)-( 465, 249)  44x 35
  [ 8] flower       ( 385, 266)-( 431, 322)  46x 56
  [ 9] flower       ( 323, 283)-( 369, 320)  46x 37
  [10] flower       ( 370, 309)-( 418, 363)  48x 54
  [11] flower       ( 429, 422)-( 468, 461)  39x 39
  [12] flower       ( 382, 352)-( 430, 400)  48x 48
  [13] flower       ( 459, 462)-( 500, 520)  41x 58
  [14] flower       ( 233, 386)-( 279, 425)  46x 39
  [15] flower       ( 253, 444)-( 305, 490)  52x 46
  [16] flower       ( 186, 485)-( 244, 533)  58x 48
  [17] flower       (  97, 439)-( 138, 478)  41x 39
  [18] flower       ( 379, 153)-( 414